### 18 - Model Deployment


This notebook prepares and deploys the registered Product Defect classifier
as a Databricks Model Serving endpoint.

The selected model has already been:

- trained and fine-tuned;
- evaluated;
- analyzed using Explainable AI;
- assessed using Responsible AI principles;
- tracked using MLflow;
- packaged with a model signature and input example;
- registered in Unity Catalog as Version 1.

This notebook does not retrain the model.

The goals are to understand:

1. What model deployment means.
2. The difference between a registered model and a serving endpoint.
3. The difference between the neural-network input and the external API input.
4. Where production image preprocessing should occur.
5. How reusable inference code should be structured.
6. How a registered Unity Catalog model becomes a serving endpoint.
7. How deployment configuration and model versioning support traceability.
8. Why deployment is not the same as proving production reliability.

Endpoint request/response testing will be performed separately in
`19_endpoint_testing`.

##### 2. Where are we now?

``` text
Raw images
    ↓
Preprocessing
    ↓
Training
    ↓
Fine-tuned ResNet18
    ↓
Evaluation
    ↓
XAI
    ↓
RAI
    ↓
MLflow
    ↓
Packaging
    ↓
Unity Catalog

product_defect_classifier
Version 1
READY

    ↓

★ DEPLOYMENT ★

    ↓

Serving Endpoint

    ↓

External Application

```

A registered model is stored and governed.

It is not yet a running prediction service.

##### 3. Registered model vs serving endpoint


Registered model is: dbw_agentic_ai_dev.product_defect_ai.product_defect_classifier Version 1

It gives us a managed model identity.

But suppose another application wants: "Classify this casting image."

It cannot simply communicate with the Unity Catalog entry.

We need a running service:

``` text

Application
     ↓
Request
     ↓
Serving Endpoint
     ↓
Model
     ↓
Prediction
     ↓
Response

```
Think:

REGISTERED MODEL --> "What model/version do we manage?"

SERVING ENDPOINT --> "Where can an application send prediction requests?"

##### 4. What actually gets deployed?

Our ResNet itself expects: Tensor [B, 3, 224, 224]

For example: [1, 3, 224, 224]

But the real-world object is: casting image.jpeg

Those are not the same thing.

Something has to perform:

``` text

JPEG
 ↓
decode image
 ↓
RGB
 ↓
resize/crop
 ↓
tensor
 ↓
normalize
 ↓
[1, 3, 224, 224]
 ↓
ResNet18

```

The question is: Who should be responsible for that preprocessing?

##### 5. Option A — client performs preprocessing

One possible design is:

``` text

Client application
      ↓
open JPEG
      ↓
convert RGB
      ↓
resize/crop
      ↓
normalize
      ↓
create [1,3,224,224]
      ↓
endpoint
      ↓
ResNet18

```

Then the endpoint only receives numbers.

This is close to what our currently registered raw PyTorch model expects.

But there's a problem.

Every caller must know:

- Which resize?
- Which crop?
- Which normalization?
- Which channel ordering?
- RGB or BGR?
- Which tensor shape?

Suppose:

Application A --> uses correct ResNet preprocessing

Application B --> forgets normalization

Application C --> uses different resizing

They are all calling the same model, but they may get very different behavior.

That's not an interface I want us to design as our final application boundary.

##### 6. Option B — preprocessing belongs with inference

A cleaner external interface is:

``` text

Application
     ↓
image
     ↓
Inference Service
     │
     ├── validate
     ├── decode
     ├── convert RGB
     ├── preprocess
     ├── tensor
     ├── model
     ├── softmax
     └── class mapping
     ↓
Prediction

```

Then the application can conceptually ask:

Here is an image.

What class is it?

instead of:

Here is a normalized
float32 [1,3,224,224]
array produced using your
specific ResNet preprocessing.

This is why we've kept: src/image_preprocessing.py

and planned: src/inference.py

##### 7. Model vs inference pipeline

This distinction is important.

The model does approximately:

``` text

tensor
  ↓
ResNet18
  ↓
logits

```

The inference pipeline does more:

``` text

image
  ↓
validate
  ↓
preprocess
  ↓
model
  ↓
logits
  ↓
softmax
  ↓
class index
  ↓
class name
  ↓
confidence

```
So:

- MODEL is one component
- INFERENCE PIPELINE is the complete prediction process

This concept applies beyond computer vision.

In your NLP project:

``` text

raw text
   ↓
tokenization / embedding
   ↓
classifier
   ↓
class

Here:

raw image
   ↓
image preprocessing
   ↓
ResNet
   ↓
class

```
Same architectural idea.

##### 8. Why src/inference.py?

We don't want this logic copied into:

- Notebook 18
- Notebook 19
- Streamlit app
- future batch job
- future API client

because eventually: Notebook preprocessing ≠ Application preprocessing

could happen.

Instead:

``` text

src/
├── image_preprocessing.py
├── vision_model.py
├── model_evaluation.py
├── explainability.py
├── responsible_ai.py
└── inference.py

```
The notebook orchestrates.

Reusable inference behavior belongs in source code.

That matches the production-style architecture we've used throughout your projects.

##### 9. What should inference.py return?

The raw neural network returns: logits

For example: [6.3, -1.8]

That's useful internally, but not a good business-facing response.

Instead we want something like:

- predicted_class = def_front
- predicted_class_index = 0
- confidence = 0.9997

Conceptually:

``` text

logits
  ↓
softmax
  ↓
probabilities
  ↓
argmax
  ↓
class index
  ↓
class name

```
Remember:

- logits → raw model scores
- softmax → class probabilities
- argmax → selected class

##### 10. Confidence still does not mean correctness

This connects directly to our RAI work.

We already observed a defect image where ResNet predicted:

- Actual: def_front
- Predicted: ok_front
- Confidence: ~98.46%

Therefore the production response may contain: confidence = 0.9846

but that means: The model strongly preferred this class relative to its alternatives.

It does not mean:

There is a 98.46% guarantee that the prediction is correct.

That's why we should call the value model confidence/probability and not a reliability guarantee.

##### 11. Why reuse build_pretrained_eval_transform()?


This is very important.

We do not write another transformation such as:

- transforms.Resize(...)
- transforms.Normalize(...)

inside inference.py.

Instead:

from src.image_preprocessing import (
    build_pretrained_eval_transform,
)

Why?

Because:

``` text

Evaluation preprocessing
             │
             ├──────────────┐
             ↓              ↓
        Evaluation       Inference
        
```
should come from the same reusable definition.

That reduces training-serving skew.

**⚖️ RAI — Reliability**

A correctly trained model can still produce unreliable predictions if deployment uses different preprocessing.

So this isn't just code cleanliness.

It's a reliability control.

##### 12. Test src/inference.py locally before deployment

This is an important engineering rule:

Don't use the serving endpoint as the first place you discover that your inference code is broken.

We should first verify:

``` text

registered model
+
inference module
+
real test image

```
inside Databricks.

In [0]:
import base64

import mlflow
import pandas as pd
import torch
import torchvision

from mlflow import MlflowClient
from mlflow.models import infer_signature
from PIL import Image

from src.inference import (
    predict_image,
)

from src.project_config import (
    REGISTERED_MODEL_NAME,
)

In [0]:
MODEL_VERSION = "1"

REGISTERED_MODEL_URI = (
    f"models:/{REGISTERED_MODEL_NAME}"
    f"/{MODEL_VERSION}"
)

print(
    "Model URI:",
    REGISTERED_MODEL_URI,
)

##### 13. Load Version 1

In [0]:
model = mlflow.pytorch.load_model(
    REGISTERED_MODEL_URI
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
model.eval()

print(
    "Device:",
    device,
)

##### 15. Recreate the deterministic data reference

In [0]:
# We need a real image path and the class mapping.Use our existing reusable module:

In [0]:
from src.data_preparation import (
    create_image_dataloaders,
)

data = create_image_dataloaders(
    preprocessing_mode="pretrained"
)

print(
    "Classes:",
    data.class_names,
)

print(
    "Class mapping:",
    data.class_to_idx,
)

##### 16. Select one raw test image

In [0]:
test_dataset = (
    data.test_loader.dataset
)

image_path, actual_class_index = (
    test_dataset.samples[0]
)

actual_class = data.class_names[
    actual_class_index
]

print(
    "Image:",
    image_path,
)

print(
    "Actual class:",
    actual_class,
)

In [0]:
image = Image.open(
    image_path
)

Notice the difference.

We're passing the raw PIL image to our inference function.

We're not manually preprocessing it in the notebook.

##### 17. Run local production-style inference

In [0]:
prediction = predict_image(
    model=model,
    image=image,
    class_names=data.class_names,
    device=device,
)

print(
    "Actual class:",
    actual_class,
)

print(
    "Predicted class:",
    prediction.predicted_class,
)

print(
    "Class index:",
    prediction.predicted_class_index,
)

print(
    "Confidence:",
    prediction.confidence,
)

print(
    "Probabilities:",
    prediction.probabilities,
)

- And now this confirms the inference pipeline works locally with the registered ResNet18 Version 1 on a real image. That's the right checkpoint before packaging it for remote serving.

##### 18. Why do this before creating the endpoint?

**Question 1**

Does our inference code work?

``` text

Registered Model v1
        +
Raw Image
        +
src/inference.py
        ↓
Prediction

```

**Question 2**

Can Databricks expose this as a remote service?

``` text

Serving Endpoint
        ↓
Network Request
        ↓
Prediction

```

If we combine both immediately and something fails, we don't know whether the problem is:

- model
- preprocessing
- image decoding
- class mapping
- serving configuration
- request format
- endpoint

Testing locally first reduces the debugging surface.

##### 19. Important deployment design issue


Our registered Unity Catalog model is currently the PyTorch ResNet model.

Its MLflow signature describes something like:

``` text

numeric tensor
[B,3,224,224]
       ↓
model
       ↓
[B,2]

```
But the external API we actually want is conceptually:

``` text

encoded/raw image
       ↓
preprocessing
       ↓
ResNet
       ↓
business response

```
Those are two different model boundaries.

This is why simply deploying Version 1 directly would expose the lower-level neural-network interface rather than the cleaner image-classification interface we're designing.

##### 20. Serving wrapper concept

For production deployment, we can wrap the underlying ResNet in an MLflow inference model.

Conceptually:

``` text

              SERVING MODEL

Request
  ↓
image representation
  ↓
decode
  ↓
PIL image
  ↓
src/inference.py
  ↓
preprocess
  ↓
ResNet18 v1
  ↓
probabilities
  ↓
class
  ↓
structured response

```

MLflow supports this kind of serving boundary through a Python model interface.

Conceptually:

``` text

class ProductDefectServingModel(
    mlflow.pyfunc.PythonModel
):
    ...

```
This wrapper owns the external contract.

##### 21. Why not deploy Version 1 directly?

We technically could expose its tensor interface.

Then a caller would have to send something conceptually like:

[
  [
    [
      [0.12, 0.14, ...],
      ...
    ],
    ...
  ]
]

for a [1,3,224,224] tensor.

That's:

1 × 3 × 224 × 224 = 150,528

floating-point values for a single image.

More importantly, the caller must reproduce the correct preprocessing.

That's not the external API I want for our portfolio project.

Instead:

``` text

Client
  ↓
image representation
  ↓
endpoint owns preprocessing

```
is a cleaner architecture.

##### 22. Two models will now have different roles


MODEL A
────────────────────────

- product_defect_classifier Version 1
- Role: Core neural network
- Input:preprocessed tensor
- Output:logits

Then we'll package:

MODEL B
────────────────────────

Serving wrapper

- Role: Production inference interface
- Input: image representation
- Internally: preprocessing + Model A behavior
- Output: class + confidence

Model B isn't a newly trained classifier.

It is a serving package around the trained classifier.

##### 23. Does this mean we're training again?


Absolutely no:

- loss.backward()
- optimizer.step()
- epochs

The flow is:

``` text

ResNet18 Version 1
       ↓
reuse weights
       ↓
wrap with inference logic
       ↓
package for serving

```
Compare:

TRAINING --> changes weights

versus:

PACKAGING --> changes interface

#### 18. Model Deployment — Part 2


##### 20. What we are building now

We currently have:

Registered Version 1
────────────────────

``` text

Input:
preprocessed tensor
[1, 3, 224, 224]

       ↓

ResNet18

       ↓

Output:
logits [1, 2]

```

But we want the endpoint boundary to be:

``` text

Application
    ↓
base64 image string
    ↓
SERVING WRAPPER
    ├── decode image
    ├── RGB conversion
    ├── ResNet preprocessing
    ├── ResNet18 prediction
    ├── softmax
    └── class mapping
    ↓
predicted_class
confidence
probabilities

```
Why base64? JSON APIs cannot conveniently carry a PIL image object. Base64 converts the image bytes into text that can be transported in JSON.

Conceptually:

``` text

JPEG file
   ↓
binary bytes
   ↓
Base64 encoding
   ↓
text string
   ↓
JSON request

```

The model is not learning from base64. Base64 is only transportation.

##### 21. One important model-version decision

We should not replace Version 1.

Version 1 has a useful identity: Version 1 = core fine-tuned ResNet18 model

Instead, we'll create a new serving-oriented model version:

``` text

Version 1
Core ResNet18
tensor → logits

        ↓ wrap same trained weights

Version 2
Serving package
base64 image → prediction response

```

No retraining occurs.

This also gives us good lineage:

``` text

Training Run
3f9b44...
     ↓
Core Model
Version 1
     ↓
Serving Packaging Run
     ↓
Serving Model
Version 2
     ↓
Endpoint

```
That is a very clean MLOps story.

##### 23. Why image.load()?

Image.open() can load image data lazily.

Since the source is an in-memory BytesIO object, explicitly calling: image.load()

ensures the image contents are read while that byte buffer is available.

Then: .convert("RGB") ensures the model consistently receives three channels.

##### 24. Create the MLflow serving wrapper in inference

##### 25. Why CPU?

You may notice in the above ML flow serving wrapper in inference code that we use

self.device = torch.device("cpu")

instead of: cuda

For our small ResNet18 image-classification project, CPU serving is a sensible initial deployment choice.

We're not saying: ResNet should always use CPU.

We're saying: Start with the simpler/lower-cost serving configuration appropriate for this project, then measure latency and throughput before deciding GPU is necessary.

That's an important production principle:

- Don't choose expensive compute because training used a GPU.
- Choose serving compute based on inference requirements.

##### 26. What does the wrapper actually contain?

``` text

ProductDefectServingModel
│
├── trained ResNet18
│      └── same learned weights
│
├── class names
│      ├── def_front
│      └── ok_front
│
├── base64 decoder
│
├── preprocessing
│      └── build_pretrained_eval_transform()
│
├── softmax
│
├── argmax
│
└── structured output

```

So we're packaging the inference pipeline, not training another model.

##### 27. Test the wrapper locally before logging it

In [0]:
import base64
import pandas as pd

from src.inference import (
    ProductDefectServingModel,
)

In [0]:
with open(
    image_path,
    "rb",
) as image_file:
    image_base64 = base64.b64encode(
        image_file.read()
    ).decode("utf-8")

In [0]:
print(
    "Base64 characters:",
    len(image_base64),
)

###### 28. Create a request exactly like the serving model expects

In [0]:
example_request = pd.DataFrame({
    "image_base64": [
        image_base64
    ]
})

display(
    example_request[
        ["image_base64"]
    ].assign(
        image_base64=lambda df:
            df["image_base64"].str[:50]
            + "..."
    )
)

##### 29. Test the wrapper

In [0]:
serving_model = (
    ProductDefectServingModel(
        model=model,
        class_names=data.class_names,
    )
)

serving_model.load_context(
    context=None
)

local_serving_prediction = (
    serving_model.predict(
        context=None,
        model_input=example_request,
    )
)

display(
    local_serving_prediction
)

##### 30. Why test again?

Because we've introduced a new component:

``` text


Base64
   ↓
decoder

Previously we tested:

PIL image
   ↓
predict_image()

Now we're testing:

base64
   ↓
decoder
   ↓
PIL
   ↓
predict_image()

```

Again, we're reducing the debugging surface before deployment.

##### 31. Create a serving signature

Now our input/output contract has changed.

Version 1 signature was approximately:

``` text

INPUT
float tensor
[-1,3,224,224]

OUTPUT
float logits
[-1,2]

```

The serving model should instead describe:

INPUT

image_base64 : string


OUTPUT

``` text

predicted_class_index : integer
predicted_class       : string
confidence            : double
defect_probability    : double
ok_probability        : double

```

Use the actual request and prediction:

In [0]:
from mlflow.models import (
    infer_signature,
)

serving_signature = (
    infer_signature(
        example_request,
        local_serving_prediction,
    )
)

print(
    serving_signature
)

##### 32. Why keep probabilities for both classes?

Suppose the endpoint returns:

- predicted_class: def_front
- confidence: 0.78

It's useful to also know:

- defect_probability:0.78
- ok_probability:0.22

Especially later if a business wants a decision policy such as:

``` text

model prediction
       ↓
decision rule
       ↓
automatic action / human review

```
But we're not setting a production threshold now.

That's an RAI/governance decision that depends on actual operational costs and requirements.

##### 33. Log the serving package

Now we package the wrapper.

First make sure no accidental MLflow run remains active:

In [0]:
if mlflow.active_run() is not None:
    mlflow.end_run()

In [0]:
serving_packaging_run = (
    mlflow.start_run(
        run_name=(
            "resnet18_serving_package"
        )
    )
)

serving_packaging_run_id = (
    serving_packaging_run.info.run_id
)

print(
    "Serving Packaging Run ID:",
    serving_packaging_run_id,
)

In [0]:
import mlflow
from mlflow import MlflowClient

from src.project_config import (
    REGISTERED_MODEL_NAME,
    SERVING_ENDPOINT_NAME,
)

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

client = MlflowClient()

SOURCE_MODEL_VERSION = "1"

source_model_version = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=SOURCE_MODEL_VERSION,
    )
)

In [0]:
print(
    "Source model version:",
    source_model_version.version,
)

print(
    "Source model run ID:",
    source_model_version.run_id,
)

print(
    "Source model tags:",
    source_model_version.tags,
)

In [0]:
RESNET18_RUN_ID = (
    source_model_version.tags.get(
        "source_training_run_id"
    )
)

if not RESNET18_RUN_ID:
    raise ValueError(
        "Registered source model version "
        "does not contain the "
        "'source_training_run_id' "
        "lineage tag."
    )

print(
    "Source training run ID:",
    RESNET18_RUN_ID,
)

In [0]:
SOURCE_PACKAGING_RUN_ID = (
    source_model_version.run_id
)

In [0]:
mlflow.set_tags({
    "model_type": (
        "product_defect_serving_wrapper"
    ),
    "purpose": (
        "databricks_model_serving"
    ),
    "source_training_run_id": (
        RESNET18_RUN_ID
    ),
    "source_packaging_run_id": (
        SOURCE_PACKAGING_RUN_ID
    ),
    "source_registered_model": (
        REGISTERED_MODEL_NAME
    ),
    "source_registered_model_version": (
        SOURCE_MODEL_VERSION
    ),
})

Notice our lineage is getting richer:

``` text

Serving Packaging Run
      │
      ├── source_training_run_id
      │       → 3f9b44...
      │
      ├── source_registered_model
      │       → product_defect_classifier
      │
      └── source_registered_model_version
              → 1

```

##### 34. Log the PyFunc model

In [0]:
serving_model_info = (
    mlflow.pyfunc.log_model(
        name="model",
        python_model=serving_model,
        signature=serving_signature,
        input_example=example_request,
        code_paths=[
            "src",
        ],
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"torch=={torch.__version__}",
            f"torchvision=={__import__('torchvision').__version__}",
            f"pandas=={pd.__version__}",
            "Pillow",
        ],
    )
)

In [0]:
print(
    "Serving Model URI:",
    serving_model_info.model_uri,
)

In [0]:
mlflow.end_run()

##### 35. Why include src?

Our wrapper depends on:

``` text

src/inference.py
        ↓
src/image_preprocessing.py
        ↓
src/project_config.py

```

The endpoint doesn't magically have your Git repository notebook environment.

So the serving package needs access to its code dependencies.

This is the deployment equivalent of something you've already learned:

Source code is part of the deployable model system.

##### 36. Test the logged model before registration

In [0]:
logged_serving_model = (
    mlflow.pyfunc.load_model(
        serving_model_info.model_uri
    )
)

In [0]:
logged_prediction = (
    logged_serving_model.predict(
        example_request
    )
)

display(
    logged_prediction
)

``` text
Original inference
       ↓
def_front

Local wrapper
       ↓
def_front

Logged MLflow wrapper
       ↓
def_front

```

##### 37. Register the serving package

In [0]:
mlflow.set_registry_uri(
    "databricks-uc"
)

In [0]:
serving_registered_model = (
    mlflow.register_model(
        model_uri=(
            serving_model_info.model_uri
        ),
        name=REGISTERED_MODEL_NAME,
    )
)

In [0]:
SERVING_MODEL_VERSION = (
    str(
        serving_registered_model.version
    )
)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME,
)

print(
    "Serving model version:",
    SERVING_MODEL_VERSION,
)

##### 38. Our registry now has two roles

product_defect_classifier

Version 1
──────────────────────────
Core ResNet18

- Input:tensor
- Output:logits


Version 2
──────────────────────────
Serving inference package

Input:base64 image

Internally:
- decode
- preprocess
- ResNet18
- softmax
- class mapping

Output: business-friendly prediction

Same learned classifier behavior.

Different packaging/interface.

##### 39. Add version description and tags

In [0]:
from mlflow import MlflowClient

client = MlflowClient()

In [0]:
serving_description = """
Serving-oriented Product Defect image classifier.

This model packages the previously trained and evaluated
fine-tuned ResNet18 classifier with production inference logic.

Input:
- base64-encoded product image

Inference pipeline:
- decode image
- convert to RGB
- deterministic pretrained ResNet18 preprocessing
- ResNet18 classification
- softmax probabilities
- class mapping

Output:
- predicted class index
- predicted class
- confidence
- defect probability
- OK probability

Classes:
- def_front
- ok_front

Lineage:
- source registered model version: 1
- underlying classifier weights are unchanged
- no retraining was performed during serving packaging

Responsible AI:
- confidence is not a guarantee of correctness
- three defect false negatives were observed during evaluation
- exact development/test duplicates were identified in the supplied dataset
- production robustness across changing image conditions has not been established
""".strip()

In [0]:
client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=SERVING_MODEL_VERSION,
    description=serving_description,
)

In [0]:
serving_version_tags = {
    "task": "image_classification",
    "architecture": "resnet18",
    "interface": "base64_image",
    "deployment_role": "serving_wrapper",
    "source_training_run_id": (
        RESNET18_RUN_ID
    ),
    "source_model_version": "1",
    "preprocessing": (
        "resnet18_pretrained_eval"
    ),
}

In [0]:
for key, value in (
    serving_version_tags.items()
):
    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=SERVING_MODEL_VERSION,
        key=key,
        value=value,
    )

Again, this metadata doesn't change predictions.

It helps us understand what this model version is and where it came from.

##### 40. Before endpoint creation: architecture checkpoint

``` text
                TRAINING
                   │
                   ▼
       Training Run 3f9b44...
                   │
                   ▼
             ResNet18
                   │
                   ▼
       Registered Version 1
       tensor → logits
                   │
                   │ same weights
                   ▼
          src/inference.py
                   │
                   ▼
       Serving Packaging Run
                   │
                   ▼
       Registered Version 2
       base64 → prediction
                   │
                   ▼
             DEPLOYMENT
                   │
                   ▼
           Serving Endpoint

```

No retraining occurred between Version 1 and the serving version.

##### 41. ⚖️ RAI — Registration still isn't production approval

Even after the serving version exists, remember:

Registered ≠ Deployed ≠ Production validated

And soon:

Endpoint READY

will mean that the serving infrastructure is ready to accept requests.

It does not mean:

- ✓ proven safe for every factory
- ✓ proven robust under every camera
- ✓ confidence calibrated
- ✓ zero missed defects
- ✓ no distribution shift

Our own evaluation already demonstrated why: a highly confident prediction can still be wrong.

In [0]:
version_2 = client.get_model_version(
    name=REGISTERED_MODEL_NAME,
    version=SERVING_MODEL_VERSION,
)

print(
    "Version:",
    version_2.version,
)

print(
    "Run ID:",
    version_2.run_id,
)

print(
    "Tags:",
    version_2.tags,
)

##### Next: Deploy Version 2

``` text

Training
   ↓
ResNet18
   ↓
Version 1 — core model
   ↓
Serving wrapper
   ↓
Version 2 — serving package
   ↓
─────────────────────────
     WE ARE HERE
─────────────────────────
   ↓
Serving Endpoint
   ↓
Notebook 19 endpoint tests

```

##### 42. What is the endpoint going to do?

The endpoint is the running infrastructure around Version 2.

``` text

                Unity Catalog
         product_defect_classifier
                 Version 2
                     │
                     │ deploy
                     ▼
        ┌─────────────────────────┐
        │ Databricks Model       │
        │ Serving Endpoint       │
        │                        │
        │ product-defect-        │
        │ classifier-endpoint    │
        └────────────┬────────────┘
                     ▲
                     │ request
                     │
               Application

Notice that we are not copying our model into Notebook 18 manually.

```

We're telling Databricks:

Run product_defect_classifier, Version 2, behind this endpoint.

##### 43. Why deploy Version 2 rather than Version 1?

Now this should be much clearer.

If we deploy: Version 1

the endpoint contract is essentially: numbers → logits

If we deploy: Version 2

the endpoint contract is:

``` text

base64 image
     ↓
useful classification response

```

For our external application architecture, Version 2 is therefore the appropriate serving artifact.

##### 44. Import the Databricks SDK

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

In [0]:
workspace_client = WorkspaceClient()

Because you're running this inside Databricks, the SDK can use the notebook's Databricks authentication context.

You may again see the familiar development warning about a notebook authentication token. That's okay for this learning/development workflow; for a production automated deployment, we'd use an appropriate service identity rather than treating notebook authentication as the final production authentication architecture.

##### 45. Inspect the deployment values first

In [0]:
print(
    "Endpoint:",
    SERVING_ENDPOINT_NAME,
)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME,
)

print(
    "Model version:",
    SERVING_MODEL_VERSION,
)

##### 46. Create the endpoint configuration

In [0]:
endpoint_config = (
    EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=(
                    REGISTERED_MODEL_NAME
                ),
                entity_version=(
                    SERVING_MODEL_VERSION
                ),
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    )
)

- entity_name --> Which registered model?
- entity_version --> Which version of that model?
- workload_size --> means we're starting with a small serving workload rather than provisioning unnecessarily large inference infrastructure.
- scale_to_zero_enabled --> True. means the endpoint can scale down when unused.For a learning project where we're sensitive to unnecessary cloud costs, that's appropriate. The tradeoff is that after being idle, the first request can take longer because the serving resources may need to become active again.

##### 47. Before creating: check whether the endpoint already exists

In [0]:
existing_endpoint = None

try:
    existing_endpoint = (
        workspace_client
        .serving_endpoints
        .get(
            SERVING_ENDPOINT_NAME
        )
    )

    print(
        "Endpoint already exists:",
        SERVING_ENDPOINT_NAME,
    )

except Exception:
    print(
        "Endpoint does not exist yet."
    )

##### 48. Create only when it doesn't exist

In [0]:
if existing_endpoint is None:

    print(
        "Creating endpoint..."
    )

    endpoint = (
        workspace_client
        .serving_endpoints
        .create_and_wait(
            name=SERVING_ENDPOINT_NAME,
            config=endpoint_config,
        )
    )

    print(
        "Endpoint created."
    )

else:

    endpoint = existing_endpoint

    print(
        "Using existing endpoint."
    )

``` text

Request endpoint creation
          ↓
Databricks provisions serving resources
          ↓
loads Version 2
          ↓
loads Python environment
          ↓
loads model
          ↓
starts serving
          ↓
endpoint becomes ready

```

**49. Important note about rerunning this notebook**

There's an important production nuance.

The previous code means:

``` text

Endpoint doesn't exist
        ↓
create Version 2 endpoint

Endpoint already exists
        ↓
leave it alone

```

It does not mean:

``` text

Endpoint already exists
        ↓
automatically replace its model

```
That's intentional.

Creating an endpoint and updating an existing production endpoint are different operations.

We don't want a notebook rerun silently changing a deployed model.

Later, if we create Version 3, we should explicitly perform an endpoint update:

``` text

Endpoint
Version 2
   ↓
controlled update
   ↓
Version 3

```

rather than treating that as accidental side effect of rerunning a notebook.

##### 50. Retrieve endpoint status

In [0]:
endpoint_status = (
    workspace_client
    .serving_endpoints
    .get(
        SERVING_ENDPOINT_NAME
    )
)

print(
    "Endpoint:",
    endpoint_status.name,
)

print(
    "State:",
    endpoint_status.state,
)

print(
    "Config:",
    endpoint_status.config,
)

##### 51. What does READY mean here?

Be careful because we've now encountered "ready" in two different places.

Earlier:

Model Version 2
Status = READY

meant: The registered model version was successfully created and is available in the registry.

Now:

Serving Endpoint READY

means: The serving infrastructure has successfully loaded the configured model and can accept inference requests.

Those are different readiness states.

``` text

REGISTERED MODEL READY
        ↓
model artifact available

ENDPOINT READY
        ↓
running service available

```

##### 52. ⚖️ Responsible AI reminder

And neither one means:

``` text

MODEL VERSION READY
        ≠
PRODUCTION SAFETY PROVEN

ENDPOINT READY
        ≠
PRODUCTION SAFETY PROVEN

```
READY is an infrastructure/lifecycle status.

Our RAI conclusions remain:

- 3 known defect false negatives
- one high-confidence wrong prediction
- development/test duplicate contamination identified
- near-duplicate risk not completely ruled out
- robustness under different cameras/lighting
- not established

That distinction is extremely important in real AI systems.

##### 53. What happens after endpoint creation?

Once the endpoint is ready, our architecture becomes:

``` text

                  APPLICATION
                       │
                       │ Base64 image
                       ▼
        ┌───────────────────────────┐
        │ Databricks Serving       │
        │ Endpoint                 │
        │                          │
        │ product-defect-          │
        │ classifier-endpoint      │
        └─────────────┬─────────────┘
                      │
                      ▼
               Model Version 2
                      │
                decode Base64
                      │
                   PIL/RGB
                      │
                 preprocess
                      │
                  ResNet18
                      │
                    logits
                      │
                   softmax
                      │
                class mapping
                      │
                      ▼
               JSON response

```
This is now very different from simply: model(image_tensor)

You've gone from building a neural network to building a deployable AI inference system.

In [0]:
import base64

import mlflow
import pandas as pd
import torch
import torchvision

from mlflow import MlflowClient
from mlflow.models import infer_signature
from PIL import Image

In [0]:
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("MLflow:", mlflow.__version__)
print("Pandas:", pd.__version__)

In [0]:
def serving_package_version(
    version: str,
) -> str:
    """
    Remove local build metadata such as '+cpu'.

    Example:
        2.7.0+cpu -> 2.7.0
    """
    return version.split("+")[0]


TORCH_VERSION = serving_package_version(
    torch.__version__
)

TORCHVISION_VERSION = serving_package_version(
    torchvision.__version__
)

print(
    "Notebook Torch:",
    torch.__version__,
)

print(
    "Serving Torch:",
    TORCH_VERSION,
)

print(
    "Notebook Torchvision:",
    torchvision.__version__,
)

print(
    "Serving Torchvision:",
    TORCHVISION_VERSION,
)

In [0]:
if mlflow.active_run() is not None:
    mlflow.end_run()


correction_run = mlflow.start_run(
    run_name=(
        "resnet18_serving_package_"
        "dependency_correction"
    )
)

CORRECTION_RUN_ID = (
    correction_run.info.run_id
)

print(
    "Correction packaging run ID:",
    CORRECTION_RUN_ID,
)

In [0]:
mlflow.set_tags({
    "model_type": (
        "product_defect_serving_wrapper"
    ),
    "purpose": (
        "databricks_model_serving"
    ),
    "source_training_run_id": (
        RESNET18_RUN_ID
    ),
    "source_registered_model": (
        REGISTERED_MODEL_NAME
    ),
    "source_registered_model_version": (
        SOURCE_MODEL_VERSION
    ),
    "supersedes_serving_version": "2",
    "packaging_change": (
        "remove_cpu_build_suffix_"
        "from_torch_dependencies"
    ),
})

In [0]:
corrected_model_info = (
    mlflow.pyfunc.log_model(
        name="model",
        python_model=serving_model,
        signature=serving_signature,
        input_example=example_request,
        code_paths=[
            "src",
        ],
        pip_requirements=[
            (
                f"mlflow=="
                f"{mlflow.__version__}"
            ),
            (
                f"torch=="
                f"{TORCH_VERSION}"
            ),
            (
                f"torchvision=="
                f"{TORCHVISION_VERSION}"
            ),
            (
                f"pandas=="
                f"{pd.__version__}"
            ),
            "Pillow",
        ],
    )
)

print(
    "Corrected model URI:",
    corrected_model_info.model_uri,
)

In [0]:
mlflow.end_run()

In [0]:
corrected_serving_model = (
    mlflow.pyfunc.load_model(
        corrected_model_info.model_uri
    )
)

corrected_prediction = (
    corrected_serving_model.predict(
        example_request
    )
)

display(
    corrected_prediction
)

In [0]:
corrected_registered_model = (
    mlflow.register_model(
        model_uri=(
            corrected_model_info.model_uri
        ),
        name=REGISTERED_MODEL_NAME,
    )
)

CORRECTED_SERVING_VERSION = str(
    corrected_registered_model.version
)

print(
    "Corrected serving version:",
    CORRECTED_SERVING_VERSION,
)

In [0]:
corrected_description = """
Serving-oriented Product Defect image classifier.

This version corrects the Python serving-environment dependencies
from the previous serving package.

The underlying fine-tuned ResNet18 classifier, learned weights,
image preprocessing, inference logic, serving input contract,
and prediction output contract are unchanged.

Dependency correction:
- torch local build suffix removed
- torchvision local build suffix removed

Serving dependencies:
- torch 2.7.0
- torchvision 0.22.0

Input:
- base64-encoded product image

Output:
- predicted class index
- predicted class
- confidence
- defect probability
- OK probability

Lineage:
- source core model version: 1
- supersedes serving model version: 2
- no retraining was performed

Responsible AI:
- confidence is not a guarantee of correctness
- three defect false negatives were observed during evaluation
- exact development/test duplicates were identified
- production robustness across changing image conditions has not
  been established
""".strip()

In [0]:
client.update_model_version(
    name=REGISTERED_MODEL_NAME,
    version=CORRECTED_SERVING_VERSION,
    description=corrected_description,
)

In [0]:
corrected_version_tags = {
    "task": "image_classification",
    "architecture": "resnet18",
    "interface": "base64_image",
    "deployment_role": "serving_wrapper",
    "source_training_run_id": (
        RESNET18_RUN_ID
    ),
    "source_model_version": (
        SOURCE_MODEL_VERSION
    ),
    "supersedes_version": "2",
    "preprocessing": (
        "resnet18_pretrained_eval"
    ),
    "packaging_change": (
        "dependency_correction"
    ),
}

In [0]:
for key, value in (
    corrected_version_tags.items()
):

    client.set_model_version_tag(
        name=REGISTERED_MODEL_NAME,
        version=CORRECTED_SERVING_VERSION,
        key=key,
        value=value,
    )

In [0]:
corrected_version = (
    client.get_model_version(
        name=REGISTERED_MODEL_NAME,
        version=CORRECTED_SERVING_VERSION,
    )
)

print(
    "Version:",
    corrected_version.version,
)

print(
    "Status:",
    corrected_version.status,
)

print(
    "Run ID:",
    corrected_version.run_id,
)

print(
    "Tags:",
    corrected_version.tags,
)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    ServedEntityInput,
)

workspace_client = WorkspaceClient()

In [0]:
SERVING_MODEL_VERSION = (
    CORRECTED_SERVING_VERSION
)

print(
    "Deploying model version:",
    SERVING_MODEL_VERSION,
)

In [0]:
updated_endpoint_config = (
    EndpointCoreConfigInput(
        served_entities=[
            ServedEntityInput(
                entity_name=(
                    REGISTERED_MODEL_NAME
                ),
                entity_version=(
                    SERVING_MODEL_VERSION
                ),
                workload_size="Small",
                scale_to_zero_enabled=True,
            )
        ]
    )
)

In [0]:
workspace_client.serving_endpoints.update_config_and_wait(
    name=SERVING_ENDPOINT_NAME,
    served_entities=(
        updated_endpoint_config.served_entities
    ),
)

In [0]:
endpoint_status = (
    workspace_client
    .serving_endpoints
    .get(
        SERVING_ENDPOINT_NAME
    )
)

print(
    "Config update:",
    endpoint_status.state.config_update,
)

print(
    "Ready:",
    endpoint_status.state.ready,
)